## Libraries

In [1]:
import tensorflow as tf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import random
import os
import pickle
import gc

# Reproducibility
seed = 42
os.environ['PYTHONHASHSEED'] = str(seed)
random.seed(seed)
np.random.seed(seed)
tf.random.set_seed(seed)

# Optional: disable eager execution (if using graph mode code)
#tf.compat.v1.disable_eager_execution()

# Session config
session_conf = tf.compat.v1.ConfigProto(
    intra_op_parallelism_threads=1,
    inter_op_parallelism_threads=1
)
sess = tf.compat.v1.Session(config=session_conf)

# Set this session as default for everything that follows
# No set_session needed — use a context manager instead
with sess.as_default():
    # your model/code here
    pass

tf.__version__

2025-06-09 07:33:37.501228: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-06-09 07:33:37.511118: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1749454417.522631    3084 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1749454417.526150    3084 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1749454417.536106    3084 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

'2.19.0'

In [2]:
from sklearn.preprocessing import StandardScaler

# Import keras
from keras.models import Sequential
from keras.layers import Dense, Dropout, TimeDistributed, LSTM#CuDNNLSTM
from keras.callbacks import EarlyStopping#, ModelCheckpoint, CSVLogger
from keras.initializers import glorot_normal
from keras.layers import RepeatVector
from keras.utils import plot_model

## Imports

In [3]:
current_path='/mnt/d/GitHub/WQU-Capstone/notebooks/PCA_8-5-25'
with open(f'{current_path}/PCA_selected_pairs.pickle', 'rb') as handle: 
    pairs = pickle.load(handle)
len(pairs)

15

In [4]:
os.chdir(current_path)

os.getcwd()

'/mnt/d/GitHub/WQU-Capstone/notebooks/PCA_8-5-25'

## Functions

### series_to_supervised

In [5]:
def series_to_supervised(data, index=None, n_in=1, n_out=1, dropnan=True):
    """
    Frame a time series as a supervised learning dataset.
    Arguments:
    data: Sequence of observations as a list or NumPy array.
    n_in: Number of lag observations as input (X).
    n_out: Number of observations as output (y).
    dropnan: Boolean whether or not to drop rows with NaN values.
    Returns:
        Pandas DataFrame of series framed for supervised learning.
    """
    n_vars = 1 if type(data) is list else data.shape[1]
    if index is None:
        df = pd.DataFrame(data)
    else:
        df = pd.DataFrame(data, index=index)
    cols, names = list(), list()
    # input sequence (t-n, ... t-1)
    for i in range(n_in, 0, -1):
        cols.append(df.shift(i))
        names += [('var%d(t-%d)' % (j+1, i)) for j in range(n_vars)]
    # forecast sequence (t, t+1, ... t+n)
    for i in range(0, n_out):
        cols.append(df.shift(-i))
        if i == 0:
            names += [('var%d(t)' % (j+1)) for j in range(n_vars)]
        else:
            names += [('var%d(t+%d)' % (j+1, i)) for j in range(n_vars)]
    # put it all together
    agg = pd.concat(cols, axis=1)
    agg.columns = names
    # drop rows with NaN values
    if dropnan:
        agg.dropna(inplace=True)
    return agg

### prepare_train_data

In [6]:
def prepare_train_data(spread, model_config):
    """
    :param spread: spread of the pair being considered
    :param model_config: dictionary with model parameters
    :return:
        tuple with training data
        tuple with validation data
        y_series in validation period (to compare with predictions later on)
    """
    train_val_split = model_config['train_val_split']

    scaler = StandardScaler()
    spread_norm = scaler.fit_transform(spread.values.reshape(spread.shape[0], 1))
    spread_norm = pd.Series(data=spread_norm.flatten(), index=spread.index)
    forecasting_data = series_to_supervised(list(spread_norm), spread.index, model_config['n_in'],
                                                    model_config['n_out'], dropnan=True)
    # define dataset
    if model_config['n_out'] == 1:
        X_series = forecasting_data.drop(columns='var1(t)')
        y_series = forecasting_data['var1(t)']
    elif model_config['n_out'] == 2:
        X_series = forecasting_data.drop(columns=['var1(t)', 'var1(t+1)'])
        y_series = forecasting_data[['var1(t)', 'var1(t+1)']]

    # split
    X_series_train = X_series[:train_val_split]
    X_series_val = X_series[train_val_split:]
    y_series_train = y_series[:train_val_split]
    y_series_val = y_series[train_val_split:]

    X_train = X_series_train.values
    X_val = X_series_val.values
    y_train = y_series_train.values
    y_val = y_series_val.values

    return (X_train, y_train), (X_val, y_val), y_series_val, scaler

### prepare_test_data

In [7]:
def prepare_test_data(spread, model_config, scaler):
    """
    """
    # normalize spread
    spread_norm = scaler.transform(spread.values.reshape(spread.shape[0], 1))
    spread_norm = pd.Series(data=spread_norm.flatten(), index=spread.index)
    forecasting_data = series_to_supervised(list(spread_norm), spread.index, model_config['n_in'],
                                                    model_config['n_out'], dropnan=True)
    # define dataset
    if model_config['n_out'] == 1:
        X_series_test = forecasting_data.drop(columns='var1(t)')
        y_series_test = forecasting_data['var1(t)']
    elif model_config['n_out'] == 2:
        X_series_test = forecasting_data.drop(columns=['var1(t)', 'var1(t+1)'])
        y_series_test = forecasting_data[['var1(t)', 'var1(t+1)']]

    X_test = X_series_test.values
    y_test = y_series_test.values

    return (X_test, y_test), y_series_test

### apply_encoder_decoder

In [8]:
def apply_encoder_decoder(X, y, validation_data, test_data, n_in, n_out, hidden_nodes, epochs, optimizer, loss_fct, batch_size=512):

    # reshape from [samples, timesteps] into [samples, timesteps, features]
    X = X.reshape((X.shape[0], X.shape[1], 1))

    if len(y.shape) == 1:
        y = np.expand_dims(y, axis=1)
    y = y.reshape((y.shape[0], y.shape[1], 1))

    X_val = validation_data[0].reshape((validation_data[0].shape[0], validation_data[0].shape[1], 1))

    if len(validation_data[1].shape) == 1:
        validation_data = (validation_data[0], np.expand_dims(validation_data[1], axis=1))
    y_val = validation_data[1].reshape((validation_data[1].shape[0], validation_data[1].shape[1], 1))

    X_test = test_data[0].reshape((test_data[0].shape[0], test_data[0].shape[1], 1))

    if len(test_data[1].shape) == 1:
        test_data = (test_data[0], np.expand_dims(test_data[1], axis=1))
    y_test = test_data[1].reshape((test_data[1].shape[0], test_data[1].shape[1], 1))

    # define model
    glorot_init = glorot_normal(seed=None)
    model = Sequential()

    # CuDNNLSTM provides a faster implementation on GPU
    model.add(LSTM(hidden_nodes[0], activation='relu', input_shape=(n_in, 1),  kernel_initializer=glorot_init))
    #model.add(LSTM(hidden_nodes[0], input_shape=(n_in, 1), kernel_initializer=glorot_init))
    model.add(RepeatVector(n_out))

    # CuDNNLSTM provides a faster implementation on GPU
    model.add(LSTM(hidden_nodes[1], activation='relu', return_sequences=True,  kernel_initializer=glorot_init))
    #model.add(LSTM(hidden_nodes[1], return_sequences=True, kernel_initializer=glorot_init))

    #model.add(Dropout(0.1))
    model.add(TimeDistributed(Dense(1, kernel_initializer=glorot_init)))
    model.compile(optimizer=optimizer, loss=loss_fct, metrics=['mae'])
    model.summary()
    os.makedirs(f'{current_path}/models/encoder_decoder', exist_ok=True)
    plot_model(model, to_file=f'{current_path}/models/encoder_decoder/model.png', show_shapes=True,
                show_layer_names=False)

    # fit model
    # simple early stopping
    es = EarlyStopping(monitor='val_loss', mode='min', verbose=1, patience=5, restore_best_weights=True)

    # fit model
    history = model.fit(X, y, epochs=epochs, verbose=1, validation_data=(X_val, y_val), shuffle=False,
                        batch_size=batch_size, callbacks=[es])

    # scores
    if len(history.history['loss']) < 500:
        train_score = [min(history.history['loss'])]#, min(history.history['mean_absolute_error'])]
        val_score = [min(history.history['val_loss'])]#, min(history.history['val_mean_absolute_error'])]
    else:
        train_score = [history.history['loss'][-1]]#, history.history['mean_absolute_error'][-1]]
        val_score = [history.history['val_loss'][-1]]#, history.history['val_mean_absolute_error'][-1]]

    score = {'train': train_score, 'val': val_score}

    predictions_train = model.predict(X, verbose=1)
    predictions_train = predictions_train.reshape(predictions_train.shape[0], predictions_train.shape[1])

    predictions_validation = model.predict(X_val, verbose=1)
    predictions_validation = predictions_validation.reshape(predictions_validation.shape[0], predictions_validation.shape[1])

    predictions_test = model.predict(X_test, verbose=1)
    predictions_test = predictions_test.reshape(predictions_test.shape[0], predictions_test.shape[1])

    print('------------------------------------------------------------')
    print('The mse train loss is: ', train_score[0])
    #print('The mae train loss is: ', train_score[1])
    print('The mse test loss is: ', val_score[0])
    #print('The mae test loss is: ', val_score[1])
    print('------------------------------------------------------------')

    return model, history, score, predictions_train, predictions_validation, predictions_test

### train_models

In [9]:
def train_models(pairs, model_config, model_type='encoder_decoder'):
    """
    This function trains the models for every pair identified.

    :param pairs: list with pairs and corresponding statistics
    :param model_config: dictionary with info for the model
    :return: all models
    """
    models = []
    for pair in pairs:

        # prepare train data
        spread = pair[2]['spread']
        train_data, validation_data, y_series_val, scaler = prepare_train_data(spread, model_config)
        
        # prepare test data
        spread_test = pair[2]['Y_test']-pair[2]['coint_coef']*pair[2]['X_test']
        test_data, y_series_test = prepare_test_data(spread_test, model_config, scaler)

        # train model and get predictions
        model, history, score, predictions_train, predictions_val, predictions_test = apply_encoder_decoder(
                                                                                            X=train_data[0],
                                                                                            y=train_data[1],
                                                                                            validation_data=validation_data,
                                                                                            test_data=test_data,
                                                                                            n_in=model_config['n_in'],
                                                                                            n_out=model_config['n_out'],
                                                                                            hidden_nodes=model_config['hidden_nodes'],
                                                                                            epochs=model_config['epochs'],
                                                                                            optimizer=model_config['optimizer'],
                                                                                            loss_fct=model_config['loss_fct'],
                                                                                            batch_size=model_config['batch_size']
                                                                                            )
        # validation
        # predictions_val = pd.DataFrame({'t': predictions_val.reshape(predictions_val.shape[0],
        #                                                                 predictions_val.shape[1])[:, 0],
        #                                 't+1': predictions_val.reshape(predictions_val.shape[0],
        #                                                                 predictions_val.shape[1])[:, 1]},
        #                                 index=y_series_val.index)

        # predictions_val['t'] = scaler.inverse_transform(np.array(predictions_val['t']))
        # predictions_val['t+1'] = scaler.inverse_transform(np.array(predictions_val['t+1']))

        # # test
        # predictions_test = pd.DataFrame({'t': predictions_test.reshape(predictions_test.shape[0],
        #                                                                 predictions_test.shape[1])[:, 0],
        #                                 't+1': predictions_test.reshape(predictions_test.shape[0],
        #                                                                 predictions_test.shape[1])[:, 1]},
        #                                 index=y_series_test.index)
        # predictions_test['t'] = scaler.inverse_transform(np.array(predictions_test['t']))
        # predictions_test['t+1'] = scaler.inverse_transform(np.array(predictions_test['t+1']))

        # # train
        # predictions_train = predictions_val.copy()  # not relevant, just to fill up

        # transform predictions to series
        #if model_type != 'encoder_decoders':
        predictions_train = scaler.inverse_transform(predictions_train)
        predictions_val = scaler.inverse_transform(predictions_val)
        predictions_test = scaler.inverse_transform(predictions_test)
        predictions_train = pd.Series(data=predictions_train.flatten(),
                                        index=spread[model_config['n_in']:-len(y_series_val)].index)
        predictions_val = pd.Series(data=predictions_val.flatten(), index=y_series_val.index)
        predictions_test = pd.Series(data=predictions_test.flatten(),
                                        index=spread_test[-len(test_data[1]):].index)

        # save all info
        # check epochs
        if len(history.history['val_loss']) == 500:
            epoch_stop = 500
        else:
            epoch_stop = len(history.history['val_loss']) - 5 # patience=5 10 50

        model_info = {'leg1': pair[0],
                        'leg2': pair[1],
                        'standardization_dict': 'scaler',
                        'history': history.history,
                        'score': score,
                        'epoch_stop': epoch_stop,
                        'predictions_train': predictions_train.copy(),
                        'predictions_val': predictions_val.copy(),
                        'predictions_test': predictions_test.copy()
                        }
        models.append(model_info)
        
    # append model configuration on last position
    models.append(model_config)

    return models

## Runner

In [10]:
input_dim = 24
hidden_nodes = [32, 16]
model_config = {"n_in": input_dim,
                        "n_out": 1,
                        "epochs": 500,
                        "hidden_nodes": hidden_nodes,
                        "loss_fct": "mse",
                        "optimizer": "rmsprop",
                        "batch_size": 32,
                        "train_val_split": '2023-01-01',
                        "test_init": '2024-01-01',}
models = train_models(pairs, model_config, model_type='encoder_decoder')

I0000 00:00:1749454419.487049    3084 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 9706 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3060, pci bus id: 0000:01:00.0, compute capability: 8.6
/home/ansel/miniconda3/envs/tf/lib/python3.12/site-packages/keras/src/layers/rnn/rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 32)             │         4,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ repeat_vector (RepeatVector)    │ (None, 1, 32)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 1, 16)          │         3,136 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed                │ (None, 1, 1)           │            17 │
│ (TimeDistributed)               │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 7,505 (29.32 KB)

 Trainable params: 7,505 (29.32 KB)

...

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_28 (LSTM)                  │ (None, 32)             │         4,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ repeat_vector_14 (RepeatVector) │ (None, 1, 32)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_29 (LSTM)                  │ (None, 1, 16)          │         3,136 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_14             │ (None, 1, 1)           │            17 │
│ (TimeDistributed)               │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 7,505 (29.32 KB)

 Trainable params: 7,505 (29.32 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/500
39/39 ━━━━━━━━━━━━━━━━━━━━ 4s 51ms/step - loss: 0.7550 - mae: 0.7277 - val_loss: 1.5718 - val_mae: 1.1084
Epoch 2/500
39/39 ━━━━━━━━━━━━━━━━━━━━ 3s 15ms/step - loss: 381.8853 - mae: 2.6237 - val_loss: 0.6354 - val_mae: 0.5507
Epoch 3/500
39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 3.8712 - mae: 0.6635 - val_loss: 8.6167 - val_mae: 1.4556
Epoch 4/500
39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.7871 - mae: 0.5058 - val_loss: 1.2547 - val_mae: 0.6414
Epoch 5/500
39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 3.1713 - mae: 0.6266 - val_loss: 1.9517 - val_mae: 0.7921
Epoch 6/500
39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.4025 - mae: 0.4861 - val_loss: 1.5683 - val_mae: 0.7358
Epoch 7/500
39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.6996 - mae: 0.4953 - val_loss: 0.7460 - val_mae: 0.5439
Epoch 7: early stopping
Restoring model weights from the end of the best epoch: 2.
39/39 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
8/8 ━━━━━━━━━

In [11]:
models

[{'leg1': 'HACK',
  'leg2': 'IGV',
  'standardization_dict': 'scaler',
  'history': {'loss': [0.8122052550315857,
    0.6105440855026245,
    0.5024591684341431,
    0.4578469395637512,
    0.2678263783454895,
    0.21021528542041779,
    0.1643819659948349,
    0.132712721824646,
    0.10692282021045685,
    0.08976675570011139,
    0.07762885093688965,
    0.06687889993190765,
    0.06417829543352127,
    0.05673300102353096,
    0.061015188694000244,
    0.05449627712368965],
   'mae': [0.6836819648742676,
    0.5952258110046387,
    0.5076307058334351,
    0.4655103385448456,
    0.38735777139663696,
    0.3338778614997864,
    0.29587867856025696,
    0.26165056228637695,
    0.23642978072166443,
    0.21833059191703796,
    0.20366409420967102,
    0.19121092557907104,
    0.18681028485298157,
    0.1766720861196518,
    0.18010087311267853,
    0.17306461930274963],
   'val_loss': [1.2797268629074097,
    0.6516391634941101,
    0.47988688945770264,
    0.4121232330799103,
    0

In [12]:
with open(f'{current_path}/models/encoder_decoder/models_n_in-'+str(input_dim)+'_hidden_nodes-'+str(hidden_nodes)+'.pkl', 'wb') as f:
    pickle.dump(models, f)